In [1]:
# ============================================================================
# CELL 1 — IMPORTS / CONFIG  (replaces the original imports cell)
# ============================================================================
import warnings

warnings.filterwarnings("ignore")

import os, math, json, copy, random
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import mlflow
import mlflow.pytorch
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from data_loading import *
from loss_funcs import *

# ---- reproducibility ----
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version  : {torch.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU              : {torch.cuda.get_device_name(0)}")
    torch.set_float32_matmul_precision("medium")
    torch.backends.cudnn.benchmark = True
    print("float32 matmul precision = 'medium', cudnn.benchmark = True")

PyTorch version  : 2.6.0+cu124
CUDA available   : True
GPU              : NVIDIA GeForce RTX 3090
float32 matmul precision = 'medium', cudnn.benchmark = True


In [2]:
weather_cols_all = ['temperature_2m',
       'apparent_temperature', 'dew_point_2m', 'relative_humidity_2m',
       'precipitation', 'rain', 'snowfall', 'cloud_cover', 'cloud_cover_low',
       'cloud_cover_mid', 'cloud_cover_high', 'surface_pressure',
       'wind_speed_10m', 'wind_direction_10m', 'wind_gusts_10m',
       'shortwave_radiation', 'diffuse_radiation', 'direct_normal_irradiance']

other_cols = [ # these are not static
    'dam_price', 'buy_bm_price', 'sell_bm_price',
    'max_power', 'max_solar', 'max_ev'
]

cat_columns = [
    'eic_code', 'dso_desc', 'station_type', 'oblast',
    'Month', 'Day', 'Hour', 'day_of_week', 'season'
]

static_cols = [
    'latitude', 'longitude', 'eic_code', 'dso_desc', 'station_type', 'oblast'
]

calendar_cols = [
    'Month', 'Day', 'Hour', 'day_of_week', 'season'
]

time_cols = ['datetime', 'time_idx']

FUTURE_REALS = weather_cols_all + calendar_cols + static_cols + other_cols
print(f"y col is: {Y_COL}, group col is: {GROUP_COL}\n"
      f"features: {FUTURE_REALS}")

y col is: sum_of_kWh, group col is: eic_code
features: ['temperature_2m', 'apparent_temperature', 'dew_point_2m', 'relative_humidity_2m', 'precipitation', 'rain', 'snowfall', 'cloud_cover', 'cloud_cover_low', 'cloud_cover_mid', 'cloud_cover_high', 'surface_pressure', 'wind_speed_10m', 'wind_direction_10m', 'wind_gusts_10m', 'shortwave_radiation', 'diffuse_radiation', 'direct_normal_irradiance', 'Month', 'Day', 'Hour', 'day_of_week', 'season', 'latitude', 'longitude', 'eic_code', 'dso_desc', 'station_type', 'oblast', 'dam_price', 'buy_bm_price', 'sell_bm_price', 'max_power', 'max_solar', 'max_ev']


In [3]:
# this data has a data column and categorical columns are kept intact and will need to be handled.
# "time_idx" is already built in train, val and test and is continuous through them
print("Loading train …")
train = load_and_prepare(TRAIN_PATH_WITH_DATETME)

print("Loading val   …")
val = load_and_prepare(VAL_PATH_WITH_DATETME)

print("Loading test  …")
test = load_and_prepare(TEST_PATH_WITH_DATETME)

print(f"train: {train.shape}")
print(f"val : {val.shape}")
print(f"test: {test.shape}")

Loading train …
Loading val   …
Loading test  …
train: (4586151, 38)
val : (293880, 38)
test: (295430, 38)


In [4]:
# ============================================================================
# CELL 3 — ROSE HYPERPARAMETERS  (v2: full feature set)
# ----------------------------------------------------------------------------
# Now feeds:
#   * TIME-VARYING NUMERIC channels  (weather, prices, max_*) -- channel-indep
#   * TIME-VARYING CYCLIC calendar   (Hour, day_of_week, Month encoded sin/cos)
#   * CATEGORICAL TIME-VARYING       (Hour, day_of_week, Month, season) -> embeddings
#   * STATIC CATEGORICAL             (eic_code, dso_desc, station_type, oblast) -> embeddings
#   * STATIC NUMERIC                 (latitude, longitude, max_power, max_solar, max_ev)
# ============================================================================
MAX_EPOCHS = 30

ROSE_CFG = dict(
    # Window sizing -- 2-week lookback captures two weekly cycles
    seq_len      = 336,
    pred_len     = 48,
    patch_len    = 16,
    patch_stride = 8,
    # Transformer
    d_model      = 128,
    n_heads      = 8,
    n_layers     = 3,
    d_ff         = 256,
    dropout      = 0.1,
    # Register
    register_size = 64,
    register_topk = 4,
    # Static encoder
    static_dim    = 64,
    cat_emb_dim   = 16,
    # Training
    batch_size   = 256,
    epochs       = MAX_EPOCHS,
    lr           = 1e-3,
    weight_decay = 1e-4,
    grad_clip    = 1.0,
    train_stride = 24,
    val_stride   = 24,
    # Loss weights -- money_pct as the main objective + small smape term
    loss_money_weight = 1.0,
    loss_smape_weight = 0.2,
    # EMA + AMP
    ema_decay = 0.999,
    use_amp   = True,
)

# ---- TIME-VARYING NUMERIC channels (continuous, fed through patch path) ----
TIMEVAR_NUMERIC = [
    *weather_cols_all,                            # 18 weather features
    "dam_price", "buy_bm_price", "sell_bm_price",
    "max_power", "max_solar", "max_ev",
]

# ---- CYCLIC-ENCODED calendar (sin/cos pairs) -- also numeric channels -----
# Month=12 next to Month=1 should be adjacent, not 11 apart -> sin/cos fixes that.
CYCLIC_CAL = {
    "Hour":        24,
    "day_of_week":  7,
    "Month":       12,
}

# ---- TIME-VARYING CATEGORICAL fields -> embedding tables -------------------
TIMEVAR_CAT_COLS = ["Hour", "day_of_week", "Month", "season"]

# ---- STATIC CATEGORICAL fields -> embedding tables -------------------------
STATIC_CAT = ["eic_code", "dso_desc", "station_type", "oblast"]
# ---- STATIC NUMERIC fields -> linear projection ---------------------------
STATIC_NUM = ["latitude", "longitude", "max_power", "max_solar", "max_ev"]

# Defensive: drop anything not actually present
TIMEVAR_NUMERIC  = [c for c in TIMEVAR_NUMERIC  if c in train.columns]
TIMEVAR_CAT_COLS = [c for c in TIMEVAR_CAT_COLS if c in train.columns]
STATIC_CAT       = [c for c in STATIC_CAT       if c in train.columns]
STATIC_NUM       = [c for c in STATIC_NUM       if c in train.columns]

PRICE_COLS = ["dam_price", "sell_bm_price", "buy_bm_price"]


def _build_vocab(series_iter):
    """Build {value -> id}, with 0 reserved for <UNK>."""
    vals = pd.concat(list(series_iter)).astype(str).unique().tolist()
    vals = sorted(vals)
    return {"<UNK>": 0, **{v: i + 1 for i, v in enumerate(vals)}}


# Build vocabs from train+val+test so eval-time stations don't OOV
TIMEVAR_CAT_VOCAB = {
    col: _build_vocab([train[col], val[col], test[col]])
    for col in TIMEVAR_CAT_COLS
}
STATIC_CAT_VOCAB = {
    col: _build_vocab([train[col], val[col], test[col]])
    for col in STATIC_CAT
}

print(f"Lookback : {ROSE_CFG['seq_len']}h   Horizon : {ROSE_CFG['pred_len']}h")
print(f"Time-var numeric channels : {len(TIMEVAR_NUMERIC)}")
print(f"Cyclic calendar (sin/cos) : {list(CYCLIC_CAL.keys())} -> {2*len(CYCLIC_CAL)} channels")
print(f"Time-var categorical embs : {list(TIMEVAR_CAT_VOCAB.keys())}  "
      f"sizes={[len(v) for v in TIMEVAR_CAT_VOCAB.values()]}")
print(f"Static categorical  embs  : {list(STATIC_CAT_VOCAB.keys())}  "
      f"sizes={[len(v) for v in STATIC_CAT_VOCAB.values()]}")
print(f"Static numeric features   : {STATIC_NUM}")


Lookback : 336h   Horizon : 48h
Time-var numeric channels : 24
Cyclic calendar (sin/cos) : ['Hour', 'day_of_week', 'Month'] -> 6 channels
Time-var categorical embs : ['Hour', 'day_of_week', 'Month', 'season']  sizes=[25, 8, 13, 5]
Static categorical  embs  : ['eic_code', 'dso_desc', 'station_type', 'oblast']  sizes=[396, 28, 8, 24]
Static numeric features   : ['latitude', 'longitude', 'max_power', 'max_solar', 'max_ev']


In [5]:
# ============================================================================
# CELL 4 — DATASET (v2): statics + cyclic + categorical embeddings
# ----------------------------------------------------------------------------
# Each item now returns 8 tensors:
#   x         : [C, L]      target + tv-numerics + cyclic sin/cos  (float32)
#   x_tv_cat  : [Kcat, L]   integer ids for time-varying categoricals (int64)
#   x_st_cat  : [Sc]        integer ids for static categoricals (int64)
#   x_st_num  : [Sn]        static numerics, z-scored (float32)
#   y         : [H]         target ground truth
#   prices    : [H, 3]      dam, sell, buy (raw, NOT z-scored)
#   t_idx     : scalar      absolute time_idx of first forecast step
#   group_id  : scalar      integer station id
# ============================================================================

class ROSEWindowDataset(Dataset):
    def __init__(self, df: pd.DataFrame,
                 seq_len: int, pred_len: int, stride: int,
                 group_to_id: Dict,
                 feat_means: np.ndarray, feat_stds: np.ndarray,
                 static_num_means: np.ndarray, static_num_stds: np.ndarray):
        self.seq_len, self.pred_len = seq_len, pred_len
        self.group_to_id = group_to_id

        self.tv_num = TIMEVAR_NUMERIC
        self.cyclic = CYCLIC_CAL
        self.tv_cat = list(TIMEVAR_CAT_VOCAB.keys())
        self.st_cat = STATIC_CAT
        self.st_num = STATIC_NUM

        self.feat_means, self.feat_stds = feat_means, feat_stds
        self.st_num_means, self.st_num_stds = static_num_means, static_num_stds

        self._cache = {}
        self.index  = []

        for grp, gdf in df.sort_values([GROUP_COL, "time_idx"]).groupby(GROUP_COL):
            n = len(gdf)
            if n < seq_len + pred_len:
                continue

            tgt = gdf[Y_COL].to_numpy(dtype=np.float32)

            # Time-varying numerics: standardize using TRAIN stats
            num = gdf[self.tv_num].to_numpy(dtype=np.float32)
            num = (num - self.feat_means) / self.feat_stds

            # Cyclic encoding: sin & cos per cyclic column
            cyc_cols = []
            for col, period in self.cyclic.items():
                vals = gdf[col].to_numpy(dtype=np.float32)
                cyc_cols.append(np.sin(2 * np.pi * vals / period))
                cyc_cols.append(np.cos(2 * np.pi * vals / period))
            cyc = (np.stack(cyc_cols, axis=1)
                   if cyc_cols else np.zeros((n, 0), dtype=np.float32))

            # Time-varying categoricals -> integer ids
            if self.tv_cat:
                tv_ids = np.stack([
                    gdf[col].astype(str).map(TIMEVAR_CAT_VOCAB[col])
                            .fillna(0).to_numpy(dtype=np.int64)
                    for col in self.tv_cat
                ], axis=0)
            else:
                tv_ids = np.zeros((0, n), dtype=np.int64)

            # Static categoricals: one value per series (first row)
            row0 = gdf.iloc[0]
            st_ids = np.array(
                [STATIC_CAT_VOCAB[col].get(str(row0[col]), 0) for col in self.st_cat],
                dtype=np.int64,
            )

            # Static numerics: standardized
            st_num_raw = (np.array([row0[c] for c in self.st_num], dtype=np.float32)
                          if self.st_num else np.zeros(0, dtype=np.float32))
            st_num = ((st_num_raw - self.st_num_means) / self.st_num_stds
                      if self.st_num else st_num_raw)

            prc = gdf[PRICE_COLS].to_numpy(dtype=np.float32)
            tix = gdf["time_idx"].to_numpy(dtype=np.int64)

            self._cache[grp] = (tgt, num, cyc, tv_ids, st_ids, st_num, prc, tix)

            last_start = n - seq_len - pred_len
            for s in range(0, last_start + 1, stride):
                self.index.append((grp, s))

    def __len__(self):
        return len(self.index)

    def __getitem__(self, i):
        grp, s = self.index[i]
        tgt, num, cyc, tv_ids, st_ids, st_num, prc, tix = self._cache[grp]
        L, H = self.seq_len, self.pred_len

        x_tgt = tgt[s:s+L]                                         # [L]
        x_num = num[s:s+L].T                                       # [Cnum, L]
        x_cyc = cyc[s:s+L].T if cyc.shape[1] else np.zeros((0, L), dtype=np.float32)
        x = np.concatenate([x_tgt[None, :], x_num, x_cyc], axis=0)  # [C, L]

        x_tv_cat = (tv_ids[:, s:s+L] if tv_ids.shape[0]
                    else np.zeros((0, L), dtype=np.int64))

        y       = tgt[s+L:s+L+H]
        prices  = prc[s+L:s+L+H]
        t_start = tix[s+L]
        gid     = self.group_to_id.get(grp, -1)

        return (
            torch.from_numpy(x),                                # [C, L]    f32
            torch.from_numpy(x_tv_cat),                         # [Kcat, L] i64
            torch.from_numpy(st_ids),                           # [Sc]      i64
            torch.from_numpy(st_num),                           # [Sn]      f32
            torch.from_numpy(y),                                # [H]
            torch.from_numpy(prices),                           # [H, 3]
            torch.tensor(t_start, dtype=torch.long),
            torch.tensor(gid, dtype=torch.long),
        )


# ---- Stats from TRAIN ONLY -------------------------------------------------
_tv_num_train = train[TIMEVAR_NUMERIC].to_numpy(dtype=np.float32)
TV_NUM_MEAN = _tv_num_train.mean(axis=0)
TV_NUM_STD  = _tv_num_train.std(axis=0) + 1e-6
del _tv_num_train

# Static numerics: one row per group, then mean/std across groups
if STATIC_NUM:
    _static_train = (train.groupby(GROUP_COL)[STATIC_NUM].first()
                     .to_numpy(dtype=np.float32))
    ST_NUM_MEAN = _static_train.mean(axis=0)
    ST_NUM_STD  = _static_train.std(axis=0) + 1e-6
else:
    ST_NUM_MEAN = np.zeros(0, dtype=np.float32)
    ST_NUM_STD  = np.ones (0, dtype=np.float32)

# group_to_id (every station seen anywhere)
all_groups = sorted(set(train[GROUP_COL].unique())
                    | set(val[GROUP_COL].unique())
                    | set(test[GROUP_COL].unique()))
GROUP_TO_ID = {g: i for i, g in enumerate(all_groups)}
N_GROUPS = len(GROUP_TO_ID)


# Stitch the lookback context onto val/test from the previous split
def _stitch_context(prev_df, target_df, seq_len):
    pieces = []
    for grp, gdf in target_df.groupby(GROUP_COL):
        ctx = prev_df[prev_df[GROUP_COL] == grp].tail(seq_len)
        pieces.append(pd.concat([ctx, gdf], axis=0))
    return (pd.concat(pieces, axis=0)
              .sort_values([GROUP_COL, "time_idx"])
              .reset_index(drop=True))


val_with_ctx  = _stitch_context(train, val,  ROSE_CFG["seq_len"])
test_with_ctx = _stitch_context(val,   test, ROSE_CFG["seq_len"])

ds_kwargs = dict(
    seq_len=ROSE_CFG["seq_len"], pred_len=ROSE_CFG["pred_len"],
    group_to_id=GROUP_TO_ID,
    feat_means=TV_NUM_MEAN, feat_stds=TV_NUM_STD,
    static_num_means=ST_NUM_MEAN, static_num_stds=ST_NUM_STD,
)
train_ds = ROSEWindowDataset(train,         stride=ROSE_CFG["train_stride"], **ds_kwargs)
val_ds   = ROSEWindowDataset(val_with_ctx,  stride=ROSE_CFG["val_stride"],   **ds_kwargs)
test_ds  = ROSEWindowDataset(test_with_ctx, stride=ROSE_CFG["pred_len"],     **ds_kwargs)

print(f"Train windows: {len(train_ds):,}")
print(f"Val   windows: {len(val_ds):,}")
print(f"Test  windows: {len(test_ds):,}")

# num_workers=0 to avoid the worker-fork pickle stall on the in-memory cache
train_loader = DataLoader(train_ds, batch_size=ROSE_CFG["batch_size"],
                          shuffle=True,  num_workers=0,
                          pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=ROSE_CFG["batch_size"],
                          shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=ROSE_CFG["batch_size"],
                          shuffle=False, num_workers=0, pin_memory=True)

# Numeric channel count (target + tv-numerics + cyclic sin/cos)
N_CHANNELS = 1 + len(TIMEVAR_NUMERIC) + 2 * len(CYCLIC_CAL)
print(f"Total numeric channels (target + tv-num + cyclic sin/cos): {N_CHANNELS}")


Train windows: 184,836
Val   windows: 11,850
Test  windows: 5,925
Total numeric channels (target + tv-num + cyclic sin/cos): 31


In [6]:
# ============================================================================
# CELL 5 — DIFFERENTIABLE LOSSES  (money_pct + SMAPE)
# ----------------------------------------------------------------------------
# Two losses combined as: money_pct + 0.2 * SMAPE
#
# MoneyLoss  : faithful PyTorch port of money_pct() / money() from loss_funcs.py.
#              Piecewise linear in y_pred -> already differentiable everywhere.
# SMAPELoss  : standard symmetric MAPE in [0, 2]. Bounded -> safe partner for
#              money_pct without distorting its gradient direction.
# ============================================================================

class MoneyLoss(nn.Module):
    """Differentiable money loss matching loss_funcs.money / money_pct.

    Modes:
        'pct_batch'  -- sum(extra) / sum(min_spend) * 100   (== numpy money_pct)
        'pct_sample' -- mean(extra / min_spend) * 100       (per-sample, stable)
        'abs'        -- sum(extra)                          (== numpy money)
    """
    def __init__(self, mode: str = "pct_sample"):
        super().__init__()
        assert mode in ("abs", "pct_batch", "pct_sample")
        self.mode = mode

    def forward(self, y_true, y_pred, dam, sell, buy):
        # money_pct() divides prices by 1000 internally -- match it
        dam, sell, buy = dam / 1000.0, sell / 1000.0, buy / 1000.0
        diff  = y_pred - y_true
        over  = F.relu( diff)              # max(diff, 0)  -- over-prediction
        under = F.relu(-diff)              # max(-diff, 0) -- under-prediction
        ordered    = y_pred * dam
        min_spend  = y_true * dam
        real_spend = ordered - sell * over + buy * under
        extra      = real_spend - min_spend

        if self.mode == "pct_batch":
            return 100.0 * extra.sum() / min_spend.sum().clamp_min(1e-8)
        if self.mode == "pct_sample":
            return 100.0 * (extra / min_spend.clamp_min(1e-8)).mean()
        return extra.sum()


class SMAPELoss(nn.Module):
    """Differentiable SMAPE in [0, 2]."""
    def forward(self, y_true, y_pred):
        num = (y_pred - y_true).abs()
        den = (y_pred.abs() + y_true.abs()).clamp_min(1e-8)
        return (2.0 * num / den).mean()


# ---- Sanity check vs numpy reference --------------------------------------
with torch.no_grad():
    yt = torch.tensor([10.0, 10.0, 10.0, 10.0])
    yp = torch.tensor([10.0, 11.5, 8.5, 10.05])
    p  = torch.tensor([5000., 5000., 5000., 5000.])
    s  = torch.tensor([3000., 3000., 3000., 3000.])
    b  = torch.tensor([7000., 7000., 7000., 7000.])

    soft_pct  = MoneyLoss(mode="pct_batch")(yt, yp, p, s, b).item()
    soft_abs  = MoneyLoss(mode="abs")(yt, yp, p, s, b).item()
    soft_psam = MoneyLoss(mode="pct_sample")(yt, yp, p, s, b).item()
    hard_pct  = money_pct(yt.numpy(), yp.numpy(), p.numpy(), s.numpy(), b.numpy())
    hard_abs  = money    (yt.numpy(), yp.numpy(), p.numpy(), s.numpy(), b.numpy())

    print(f"Sanity -- pct_batch  surrogate: {soft_pct:.6f}   numpy: {hard_pct:.6f}")
    print(f"Sanity -- abs        surrogate: {soft_abs:.6f}   numpy: {hard_abs:.6f}")
    print(f"Sanity -- pct_sample (training mode, no numpy ref): {soft_psam:.6f}")


Sanity -- pct_batch  surrogate: 3.049999   numpy: 3.050000
Sanity -- abs        surrogate: 6.099998   numpy: 6.100000
Sanity -- pct_sample (training mode, no numpy ref): 3.049999


In [7]:
# ============================================================================
# CELL 6 — ROSE MODEL (v2): static encoder + tv-categorical embeddings
# ----------------------------------------------------------------------------
# Architecture changes vs v1:
#   * StaticEncoder: per-station info (eic_code, dso_desc, station_type, oblast,
#     latitude/longitude, max_power/solar/ev) -> static_dim vector that is
#     ADDED (FiLM-style) to every patch token. Gives the model an explicit
#     identity signal on top of the implicit RevIN + register selection.
#   * TimeVarCatEncoder: Hour/day_of_week/Month/season embeddings, mean-pooled
#     per patch, projected to d_model, added to patch tokens. Lets the model
#     see calendar context discretely (in addition to cyclic sin/cos channels).
#   * Numeric channels: target + tv-numerics + cyclic sin/cos as before.
# ============================================================================

class StaticEncoder(nn.Module):
    def __init__(self, cat_vocabs: Dict[str, Dict], n_static_num: int,
                 cat_emb_dim: int, static_dim: int):
        super().__init__()
        self.cat_keys = list(cat_vocabs.keys())
        self.cat_embs = nn.ModuleList([
            nn.Embedding(len(cat_vocabs[k]), cat_emb_dim) for k in self.cat_keys
        ])
        in_dim = cat_emb_dim * len(self.cat_keys) + n_static_num
        self.in_dim   = in_dim
        self.out_dim  = static_dim if in_dim > 0 else 0
        self.proj = (nn.Sequential(
                        nn.Linear(in_dim, static_dim), nn.GELU(),
                        nn.Linear(static_dim, static_dim))
                     if in_dim > 0 else None)

    def forward(self, st_cat_ids, st_num):
        # st_cat_ids: [B, n_st_cat]   st_num: [B, n_st_num]
        parts = [emb(st_cat_ids[:, i]) for i, emb in enumerate(self.cat_embs)]
        if st_num.shape[-1] > 0:
            parts.append(st_num)
        if not parts:
            return torch.zeros(st_cat_ids.size(0), 0, device=st_cat_ids.device)
        h = torch.cat(parts, dim=-1)
        return self.proj(h)


class TimeVarCatEncoder(nn.Module):
    """Embed per-timestep categoricals, average-pool per patch, project to d_model."""
    def __init__(self, cat_vocabs: Dict[str, Dict], cat_emb_dim: int,
                 patch_len: int, patch_stride: int, d_model: int):
        super().__init__()
        self.cat_keys = list(cat_vocabs.keys())
        self.cat_embs = nn.ModuleList([
            nn.Embedding(len(cat_vocabs[k]), cat_emb_dim) for k in self.cat_keys
        ])
        self.patch_len, self.patch_stride = patch_len, patch_stride
        in_dim = cat_emb_dim * len(self.cat_keys)
        self.proj = nn.Linear(in_dim, d_model) if in_dim > 0 else None

    def forward(self, x_tv_cat):
        # x_tv_cat: [B, K, L]  ints
        if self.proj is None or x_tv_cat.numel() == 0:
            return None
        embs = [emb(x_tv_cat[:, i, :]) for i, emb in enumerate(self.cat_embs)]
        h = torch.cat(embs, dim=-1)                              # [B, L, K*emb]
        h = h.transpose(1, 2)                                    # [B, K*emb, L]
        h = h.unfold(-1, self.patch_len, self.patch_stride)      # [B, K*emb, N, P]
        h = h.mean(dim=-1).transpose(1, 2)                       # [B, N, K*emb]
        return self.proj(h)                                      # [B, N, d_model]


class RevIN(nn.Module):
    """Reversible per-channel instance normalization (Kim et al., 2021)."""
    def __init__(self, num_channels: int, eps: float = 1e-5, affine: bool = True):
        super().__init__()
        self.eps, self.affine = eps, affine
        if affine:
            self.weight = nn.Parameter(torch.ones(num_channels))
            self.bias   = nn.Parameter(torch.zeros(num_channels))

    def normalize(self, x):
        self.mean = x.mean(dim=-1, keepdim=True).detach()
        self.std  = x.std(dim=-1, keepdim=True).detach() + self.eps
        x = (x - self.mean) / self.std
        if self.affine:
            x = x * self.weight.view(1, -1, 1) + self.bias.view(1, -1, 1)
        return x

    def denormalize(self, x):
        if self.affine:
            x = (x - self.bias.view(1, -1, 1)) / (self.weight.view(1, -1, 1) + self.eps)
        return x * self.std + self.mean


class TSRegister(nn.Module):
    """ROSE's Time-Series Register: learnable codebook + top-K retrieval."""
    def __init__(self, register_size: int, d_model: int, topk: int):
        super().__init__()
        self.codebook = nn.Parameter(torch.randn(register_size, d_model) * 0.02)
        self.topk     = topk
        self.q_proj   = nn.Linear(d_model, d_model, bias=False)

    def forward(self, patch_tokens):
        q  = self.q_proj(patch_tokens.mean(dim=1))
        cb = F.normalize(self.codebook, dim=-1)
        qn = F.normalize(q, dim=-1)
        sim = qn @ cb.T
        topk_sim, topk_idx = sim.topk(self.topk, dim=-1)
        weights  = F.softmax(topk_sim, dim=-1).unsqueeze(-1)
        selected = self.codebook[topk_idx]
        return selected * weights


class ROSE(nn.Module):
    def __init__(self, n_channels: int, seq_len: int, pred_len: int,
                 patch_len: int, patch_stride: int,
                 d_model: int, n_heads: int, n_layers: int, d_ff: int,
                 dropout: float, register_size: int, register_topk: int,
                 target_channel_idx: int,
                 static_encoder: StaticEncoder,
                 tv_cat_encoder: TimeVarCatEncoder):
        super().__init__()
        self.n_channels = n_channels
        self.target_idx = target_channel_idx
        self.patch_len, self.patch_stride = patch_len, patch_stride
        self.n_patches = (seq_len - patch_len) // patch_stride + 1
        self.pred_len = pred_len

        self.revin       = RevIN(n_channels)
        self.patch_embed = nn.Linear(patch_len, d_model)
        self.pos_embed   = nn.Parameter(torch.randn(1, self.n_patches, d_model) * 0.02)
        self.register    = TSRegister(register_size, d_model, register_topk)
        self.reg_pos     = nn.Parameter(torch.randn(1, register_topk, d_model) * 0.02)

        # Static + tv-cat injection
        self.static_encoder = static_encoder
        self.static_to_dm   = (nn.Linear(static_encoder.out_dim, d_model)
                               if static_encoder.out_dim > 0 else None)
        self.tv_cat_encoder = tv_cat_encoder

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, activation="gelu", norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.head_dropout = nn.Dropout(dropout)
        self.head = nn.Linear(self.n_patches * d_model, pred_len)

    def forward(self, x_num, x_tv_cat, st_cat_ids, st_num):
        B, C, L = x_num.shape
        x = self.revin.normalize(x_num)

        # Channel-independent: fold C into batch
        x_ci = x.reshape(B * C, L)
        patches = x_ci.unfold(-1, self.patch_len, self.patch_stride)
        tokens  = self.patch_embed(patches) + self.pos_embed       # [B*C, N, d]

        # --- Static injection (per-station identity, broadcast over channels)
        if self.static_to_dm is not None:
            s = self.static_to_dm(self.static_encoder(st_cat_ids, st_num))   # [B, d]
            s = s.unsqueeze(1).expand(B, C, s.size(-1)).reshape(B * C, 1, -1)
            tokens = tokens + s

        # --- Time-varying categorical injection (calendar tokens, same across channels)
        tv_cat_tokens = self.tv_cat_encoder(x_tv_cat)              # [B, N, d] or None
        if tv_cat_tokens is not None:
            tv_cat_tokens = (tv_cat_tokens.unsqueeze(1)
                                          .expand(B, C, -1, -1)
                                          .reshape(B * C, self.n_patches, -1))
            tokens = tokens + tv_cat_tokens

        reg_tokens = self.register(tokens) + self.reg_pos
        seq = torch.cat([reg_tokens, tokens], dim=1)
        seq = self.encoder(seq)
        patch_out = seq[:, reg_tokens.size(1):, :]
        flat = self.head_dropout(patch_out.reshape(B * C, -1))
        y = self.head(flat).reshape(B, C, self.pred_len)
        y = self.revin.denormalize(y)
        return y[:, self.target_idx, :]


static_enc = StaticEncoder(STATIC_CAT_VOCAB, len(STATIC_NUM),
                           ROSE_CFG["cat_emb_dim"], ROSE_CFG["static_dim"])
tv_cat_enc = TimeVarCatEncoder(TIMEVAR_CAT_VOCAB, ROSE_CFG["cat_emb_dim"],
                               ROSE_CFG["patch_len"], ROSE_CFG["patch_stride"],
                               ROSE_CFG["d_model"])

model = ROSE(
    n_channels    = N_CHANNELS,
    seq_len       = ROSE_CFG["seq_len"],
    pred_len      = ROSE_CFG["pred_len"],
    patch_len     = ROSE_CFG["patch_len"],
    patch_stride  = ROSE_CFG["patch_stride"],
    d_model       = ROSE_CFG["d_model"],
    n_heads       = ROSE_CFG["n_heads"],
    n_layers      = ROSE_CFG["n_layers"],
    d_ff          = ROSE_CFG["d_ff"],
    dropout       = ROSE_CFG["dropout"],
    register_size = ROSE_CFG["register_size"],
    register_topk = ROSE_CFG["register_topk"],
    target_channel_idx = 0,
    static_encoder = static_enc,
    tv_cat_encoder = tv_cat_enc,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"ROSE parameters: {n_params/1e6:.2f}M")
print(model)


ROSE parameters: 0.72M
ROSE(
  (revin): RevIN()
  (patch_embed): Linear(in_features=16, out_features=128, bias=True)
  (register): TSRegister(
    (q_proj): Linear(in_features=128, out_features=128, bias=False)
  )
  (static_encoder): StaticEncoder(
    (cat_embs): ModuleList(
      (0): Embedding(396, 16)
      (1): Embedding(28, 16)
      (2): Embedding(8, 16)
      (3): Embedding(24, 16)
    )
    (proj): Sequential(
      (0): Linear(in_features=69, out_features=64, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=64, out_features=64, bias=True)
    )
  )
  (static_to_dm): Linear(in_features=64, out_features=128, bias=True)
  (tv_cat_encoder): TimeVarCatEncoder(
    (cat_embs): ModuleList(
      (0): Embedding(25, 16)
      (1): Embedding(8, 16)
      (2): Embedding(13, 16)
      (3): Embedding(5, 16)
    )
    (proj): Linear(in_features=64, out_features=128, bias=True)
  )
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-2): 3 x Trans

In [8]:
# ============================================================================
# CELL 7 — TRAINING LOOP (v2): money_pct + 0.2*SMAPE, AMP, EMA
# ============================================================================
from tqdm.notebook import tqdm
from contextlib import nullcontext


def collect_metrics(y_true, y_pred, dam, sell, buy) -> Dict[str, float]:
    return dict(
        mae       = float(np.mean(np.abs(y_pred - y_true))),
        rmse      = float(rmse(y_true, y_pred)),
        mape      = float(mape(y_true, y_pred)),
        smape     = float(smape(y_true, y_pred)),
        money     = float(money(y_true, y_pred, dam, sell, buy)),
        money_pct = float(money_pct(y_true, y_pred, dam, sell, buy)),
        bias      = float(y_pred.sum() - y_true.sum()),
    )


# ---- Exponential moving average over weights ------------------------------
class EMA:
    """Tracks an EMA copy of every floating-point parameter/buffer.

    Standard trick: evaluate with EMA weights, then restore the live weights
    so training continues normally. Usually 0.3-0.8% MONEY_PCT improvement
    on validation at zero training cost.
    """
    def __init__(self, model: nn.Module, decay: float = 0.999):
        self.decay = decay
        self.shadow = {k: v.detach().clone()
                       for k, v in model.state_dict().items()
                       if v.dtype.is_floating_point}

    @torch.no_grad()
    def update(self, model: nn.Module):
        for k, v in model.state_dict().items():
            if k in self.shadow:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)

    def apply_to(self, model: nn.Module):
        self._backup = {k: v.detach().clone()
                        for k, v in model.state_dict().items()
                        if k in self.shadow}
        sd = model.state_dict()
        for k in self.shadow:
            sd[k].copy_(self.shadow[k])

    def restore(self, model: nn.Module):
        sd = model.state_dict()
        for k, v in self._backup.items():
            sd[k].copy_(v)
        del self._backup


@torch.no_grad()
def evaluate(model, loader, money_loss_fn):
    model.eval()
    ys, yps, prs, losses = [], [], [], []
    for x, x_tv_cat, st_cat, st_num, y, prices, _, _ in loader:
        x        = x.to(DEVICE)
        x_tv_cat = x_tv_cat.to(DEVICE)
        st_cat   = st_cat.to(DEVICE);   st_num = st_num.to(DEVICE)
        y        = y.to(DEVICE);        prices = prices.to(DEVICE)
        yp = model(x, x_tv_cat, st_cat, st_num)
        dam, sell, buy = prices[..., 0], prices[..., 1], prices[..., 2]
        losses.append(money_loss_fn(y, yp, dam, sell, buy).item())
        ys.append(y.cpu().numpy()); yps.append(yp.cpu().numpy()); prs.append(prices.cpu().numpy())
    y_arr  = np.concatenate(ys ).reshape(-1)
    yp_arr = np.concatenate(yps).reshape(-1)
    pr_arr = np.concatenate(prs).reshape(-1, 3)
    m = collect_metrics(y_arr, yp_arr, pr_arr[:, 0], pr_arr[:, 1], pr_arr[:, 2])
    m["loss"] = float(np.mean(losses))
    return m, y_arr, yp_arr, pr_arr


# ---- Sanity probe ---------------------------------------------------------
print("Sanity probe ...", flush=True)
batch = next(iter(train_loader))
_x, _xc, _sc, _sn, _y, _p, _, _ = batch
print(f"  x={tuple(_x.shape)}  x_tv_cat={tuple(_xc.shape)}  "
      f"st_cat={tuple(_sc.shape)}  st_num={tuple(_sn.shape)}", flush=True)
with torch.no_grad():
    _yp = model(_x.to(DEVICE), _xc.to(DEVICE), _sc.to(DEVICE), _sn.to(DEVICE))
print(f"  forward ok: yp={tuple(_yp.shape)}", flush=True)
del batch, _x, _xc, _sc, _sn, _y, _p, _yp


# ---- MLflow setup ---------------------------------------------------------
mlflow.set_experiment(f"electricity_forecasting_rose_v2_{MAX_EPOCHS}_epochs")
run = mlflow.start_run(run_name=f"ROSE_v2_seq{ROSE_CFG['seq_len']}_h{ROSE_CFG['pred_len']}")

mlflow.log_params({**ROSE_CFG,
                   "n_channels":       N_CHANNELS,
                   "n_stations":       N_GROUPS,
                   "n_train_windows":  len(train_ds),
                   "n_val_windows":    len(val_ds),
                   "n_test_windows":   len(test_ds),
                   "device":           str(DEVICE),
                   "params_M":         round(n_params/1e6, 3)})
mlflow.log_param("tv_numeric",       ",".join(TIMEVAR_NUMERIC))
mlflow.log_param("tv_cat",           ",".join(TIMEVAR_CAT_VOCAB.keys()))
mlflow.log_param("static_cat",       ",".join(STATIC_CAT))
mlflow.log_param("static_num",       ",".join(STATIC_NUM))
mlflow.log_param("cyclic_cal",       ",".join(CYCLIC_CAL.keys()))
mlflow.log_param("train_date_range", f"{train['datetime'].min()} -> {train['datetime'].max()}")
mlflow.log_param("val_date_range",   f"{val['datetime'].min()} -> {val['datetime'].max()}")
mlflow.log_param("test_date_range",  f"{test['datetime'].min()} -> {test['datetime'].max()}")


# ---- Optimizer / scheduler / loss / EMA / AMP -----------------------------
optimizer = torch.optim.AdamW(model.parameters(),
                              lr=ROSE_CFG["lr"],
                              weight_decay=ROSE_CFG["weight_decay"])
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=ROSE_CFG["lr"],
    steps_per_epoch=len(train_loader), epochs=ROSE_CFG["epochs"],
    pct_start=0.1, anneal_strategy="cos",
)
money_loss_train = MoneyLoss(mode="pct_sample").to(DEVICE)
money_loss_eval  = MoneyLoss(mode="pct_sample").to(DEVICE)
smape_loss       = SMAPELoss().to(DEVICE)
ema = EMA(model, decay=ROSE_CFG["ema_decay"])

use_amp = ROSE_CFG["use_amp"] and DEVICE.type == "cuda"
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
amp_ctx = (lambda: torch.cuda.amp.autocast()) if use_amp else (lambda: nullcontext())

best_val_money_pct = float("inf")
best_state = None
ckpt_path = "../models/rose/rose_best.pt"
os.makedirs(os.path.dirname(ckpt_path), exist_ok=True)

print(f"Starting training: {ROSE_CFG['epochs']} epochs x {len(train_loader)} batches",
      flush=True)

for epoch in tqdm(range(1, ROSE_CFG["epochs"] + 1), desc="Epochs"):
    model.train()
    running_loss = running_money = running_smape = 0.0
    batch_bar = tqdm(train_loader, leave=False, desc=f"Epoch {epoch}")
    for x, x_tv_cat, st_cat, st_num, y, prices, _, _ in batch_bar:
        x        = x.to(DEVICE, non_blocking=True)
        x_tv_cat = x_tv_cat.to(DEVICE, non_blocking=True)
        st_cat   = st_cat.to(DEVICE, non_blocking=True)
        st_num   = st_num.to(DEVICE, non_blocking=True)
        y        = y.to(DEVICE, non_blocking=True)
        prices   = prices.to(DEVICE, non_blocking=True)
        dam, sell, buy = prices[..., 0], prices[..., 1], prices[..., 2]

        with amp_ctx():
            yp = model(x, x_tv_cat, st_cat, st_num)
            if not torch.isfinite(yp).all():
                raise RuntimeError(f"NaN/Inf in predictions at epoch {epoch}")
            m_loss  = money_loss_train(y, yp, dam, sell, buy)
            sm_loss = smape_loss(y, yp)
            loss = (ROSE_CFG["loss_money_weight"] * m_loss
                  + ROSE_CFG["loss_smape_weight"] * sm_loss)

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), ROSE_CFG["grad_clip"])
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        ema.update(model)

        running_loss  += loss.item()
        running_money += m_loss.item()
        running_smape += sm_loss.item()
        batch_bar.set_postfix(loss=f"{loss.item():.4f}",
                              money=f"{m_loss.item():.3f}",
                              smape=f"{sm_loss.item():.3f}")

    n = max(1, len(train_loader))
    train_loss, train_money, train_smape = (running_loss / n,
                                            running_money / n,
                                            running_smape / n)

    # Validate with EMA weights
    ema.apply_to(model)
    val_metrics, *_ = evaluate(model, val_loader, money_loss_eval)
    ema.restore(model)

    print(f"epoch {epoch:02d} | train={train_loss:.4f} "
          f"money={train_money:.3f} smape={train_smape:.3f} | "
          f"val_money={val_metrics['money']:.2f} "
          f"val_pct={val_metrics['money_pct']:.3f}% "
          f"val_mae={val_metrics['mae']:.3f}", flush=True)

    mlflow.log_metrics({
        "train_loss":      train_loss,
        "train_money_pct": train_money,
        "train_smape":     train_smape,
        "lr":              optimizer.param_groups[0]["lr"],
        **{f"val_{k}": v for k, v in val_metrics.items()},
    }, step=epoch)

    if val_metrics["money_pct"] < best_val_money_pct:
        best_val_money_pct = val_metrics["money_pct"]
        # Save EMA weights as the checkpoint -- those are what we eval with
        ema.apply_to(model)
        best_state = copy.deepcopy(model.state_dict())
        ema.restore(model)
        torch.save(best_state, ckpt_path)
        mlflow.log_metric("best_val_money_pct", best_val_money_pct, step=epoch)

# Restore best weights for final evaluation
if best_state is not None:
    model.load_state_dict(best_state)
print(f"Best val money_pct: {best_val_money_pct:.4f}")


Sanity probe ...
  x=(256, 31, 336)  x_tv_cat=(256, 4, 336)  st_cat=(256, 4)  st_num=(256, 5)
  forward ok: yp=(256, 48)
Starting training: 30 epochs x 722 batches


Epochs:   0%|          | 0/30 [00:00<?, ?it/s]

Epoch 1:   0%|          | 0/722 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# ============================================================================
# CELL 8 — INFERENCE  (v2: handle 8-tuple batches from new dataset)
# ============================================================================

@torch.no_grad()
def predict_to_frame(model, dataset: ROSEWindowDataset, source_df: pd.DataFrame,
                     loader: DataLoader) -> pd.DataFrame:
    """Run model on `loader` and join predictions back to `source_df`."""
    model.eval()
    pred_chunks = []
    for x, x_tv_cat, st_cat, st_num, y, prices, t_start, gid in tqdm(
            loader, desc="Predicting", leave=False):
        x        = x.to(DEVICE, non_blocking=True)
        x_tv_cat = x_tv_cat.to(DEVICE, non_blocking=True)
        st_cat   = st_cat.to(DEVICE, non_blocking=True)
        st_num   = st_num.to(DEVICE, non_blocking=True)
        yp = model(x, x_tv_cat, st_cat, st_num).cpu().numpy()
        pred_chunks.append(yp)
    preds = np.concatenate(pred_chunks, axis=0)            # [n_windows, H]

    # DataLoader without shuffle preserves dataset order, so dataset.index
    # aligns 1:1 with rows of `preds`.
    H = dataset.pred_len
    rows = []
    for w_idx, (grp, s) in enumerate(dataset.index):
        tgt, num, cyc, tv_ids, st_ids, st_n, prc, tix = dataset._cache[grp]
        forecast_t = tix[s + dataset.seq_len : s + dataset.seq_len + H]
        true_y     = tgt[s + dataset.seq_len : s + dataset.seq_len + H]
        rows.append(pd.DataFrame({
            GROUP_COL: grp,
            "time_idx": forecast_t,
            Y_COL: true_y,
            "pred": preds[w_idx],
        }))
    pred_df = pd.concat(rows, ignore_index=True)
    pred_df = pred_df.drop_duplicates(subset=[GROUP_COL, "time_idx"], keep="last")

    join_cols = [GROUP_COL, "time_idx", "datetime", *PRICE_COLS]
    eval_df = pred_df.merge(source_df[join_cols],
                            on=[GROUP_COL, "time_idx"], how="left")
    return eval_df.sort_values([GROUP_COL, "time_idx"]).reset_index(drop=True)


val_eval  = predict_to_frame(model, val_ds,  val,  val_loader)
test_eval = predict_to_frame(model, test_ds, test, test_loader)

print(f"val_eval : {val_eval.shape}, NaNs={val_eval.isna().any().any()}")
print(f"test_eval: {test_eval.shape}, NaNs={test_eval.isna().any().any()}")


In [ ]:
# ============================================================================
# CELL 9 — FINAL METRICS + MLflow logging  (replaces the original eval cell)
# ============================================================================
def _prices(df):
    return df["dam_price"].values, df["sell_bm_price"].values, df["buy_bm_price"].values

def report(df, split):
    yt, yp = df[Y_COL].values, df["pred"].values
    dam, sell, buy = _prices(df)
    m = collect_metrics(yt, yp, dam, sell, buy)
    print(f"── {split} ──────────────────────────────────────────────")
    print(f"Aligned samples : {len(df):,}")
    print(f"MAE       : {m['mae']:.4f}")
    print(f"RMSE      : {m['rmse']:.4f}")
    print(f"MAPE      : {m['mape']:.2f} %")
    print(f"SMAPE     : {m['smape']:.4f}")
    print(f"MONEY     : {m['money']:.4f}")
    print(f"MONEY_PCT : {m['money_pct']:.4f}%")
    print(f"BIAS      : {m['bias']:+.2f}  (pred_sum - actual_sum)")
    return m

val_m  = report(val_eval,  "Validation")
test_m = report(test_eval, "Test")

mlflow.log_metrics({**{f"val_{k}":  v for k, v in val_m.items()},
                    **{f"test_{k}": v for k, v in test_m.items()}})

# Log artifacts
mlflow.log_artifact(ckpt_path)
val_eval.to_parquet("val_eval.parquet");  mlflow.log_artifact("val_eval.parquet")
test_eval.to_parquet("test_eval.parquet"); mlflow.log_artifact("test_eval.parquet")
mlflow.pytorch.log_model(model, "rose_model")

print(f"MLflow run logged → {mlflow.get_tracking_uri()}")

In [ ]:
val_eval

In [ ]:
# ============================================================================
# CELL 10 — PER-STATION METRICS (fixed: real money/money_pct, not duplicated MAPE)
# ============================================================================
def per_station_metrics(eval_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for grp, gdf in eval_df.groupby(GROUP_COL):
        dam, sell, buy = _prices(gdf)
        rows.append({
            GROUP_COL:   grp,
            "n":         len(gdf),
            "MAE":       float(np.mean(np.abs(gdf["pred"] - gdf[Y_COL]))),
            "RMSE":      rmse (gdf[Y_COL], gdf["pred"]),
            "MAPE":      mape (gdf[Y_COL], gdf["pred"]),
            "SMAPE":     smape(gdf[Y_COL], gdf["pred"]),
            "MONEY":     money    (gdf[Y_COL], gdf["pred"], dam, sell, buy),
            "MONEY_PCT": money_pct(gdf[Y_COL], gdf["pred"], dam, sell, buy),
        })
    return pd.DataFrame(rows).sort_values("MONEY_PCT")

test_station_metrics = per_station_metrics(test_eval)
print("Top-10 best stations (test MONEY_PCT):")
print(test_station_metrics.head(10).to_string(index=False))
print("\nBottom-10 worst stations (test MONEY_PCT):")
print(test_station_metrics.tail(10).to_string(index=False))

test_station_metrics.to_csv("test_station_metrics.csv", index=False)
mlflow.log_artifact("test_station_metrics.csv")

In [ ]:
# ============================================================================
# CELL 11 — PLOTS  (your existing plot_forecast utility — unchanged)
# ============================================================================
def plot_forecast(df, eic_code, start_dt=None, end_dt=None, title_prefix=""):
    df = df[df[GROUP_COL] == eic_code].sort_values("datetime")
    if df.empty: raise ValueError(f"No data for EiC code: {eic_code!r}")
    if start_dt is not None: df = df[df["datetime"] >= pd.Timestamp(start_dt)]
    if end_dt   is not None: df = df[df["datetime"] <= pd.Timestamp(end_dt)]
    if df.empty: raise ValueError("No data in the specified datetime range.")

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(df["datetime"], df[Y_COL],  label="True",      linewidth=1, color="steelblue")
    ax.plot(df["datetime"], df["pred"], label="Predicted", linewidth=1, color="tomato", alpha=0.85)
    ax.set_title(
        f"{title_prefix}{eic_code}  |  MAPE={mape(df[Y_COL], df['pred']):.3f}"
        f"  MONEY_PCT={money_pct(df[Y_COL], df['pred'], *_prices(df)):.2f}%"
        f"  ({df['datetime'].min().date()} – {df['datetime'].max().date()})"
    )
    ax.set_xlabel("Datetime"); ax.set_ylabel(Y_COL); ax.legend()
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
    fig.autofmt_xdate(rotation=0, ha="center"); plt.tight_layout()
    fname = f"forecast_{title_prefix.strip().strip('[]').lower() or 'plot'}_{eic_code}.png"
    plt.savefig(fname, dpi=110, bbox_inches="tight")
    mlflow.log_artifact(fname)
    plt.show()

best_station  = test_station_metrics.iloc[0][GROUP_COL]
worst_station = test_station_metrics.iloc[-1][GROUP_COL]
print(f"Best  station (MONEY_PCT): {best_station}")
plot_forecast(test_eval, eic_code=best_station,  title_prefix="[BEST] ")
print(f"Worst station (MONEY_PCT): {worst_station}")
plot_forecast(test_eval, eic_code=worst_station, title_prefix="[WORST] ")

mlflow.end_run()
print("Done.")